# Problem set 1

## Q1: Tensors and Automatic Differentiation


### Q1a

In [1]:
import torch
import numpy as np
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
from models.linear import LinearRegression

torch.manual_seed(42)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

In [2]:
#Q1a.1
a = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32).to(DEVICE)

#Q1a.2
ones = np.ones((3,3))
b = torch.tensor(ones, dtype=torch.float32).to(DEVICE)

In [3]:
#Q1a.3
a_transpose = a.unsqueeze(1)

#Q1a.4
square = torch.square(a)
square

tensor([ 1.,  4.,  9., 16., 25.])

In [4]:
#Q1a.5
b_b = b@b

b_b

tensor([[3., 3., 3.],
        [3., 3., 3.],
        [3., 3., 3.]])

### Q1b

$f(x) = x^2 + 3x + 1$ 

$df/dx = 2x + 3$

When $x = 2, df/dx = 7$

In [5]:
x = torch.tensor([2],dtype=torch.float32,  requires_grad=True)
y = x **2 + 3*x + 1
y.backward()
print(x.grad)

tensor([7.])


### Q1c

$g(x, y) = x^2y + y^3$

$dg/dx = 2xy,  dg/dy = x^2 + 3y^2$

In [6]:
x = torch.tensor([1.], requires_grad=True)
y = torch.tensor([2.], requires_grad=True)
g = x**2*y + y**3

g.backward()
print(f"partial derivate of g wrt x, {x.grad}")
print(f"patital derivate of g wrt y, {y.grad}")

partial derivate of g wrt x, tensor([4.])
patital derivate of g wrt y, tensor([13.])


##  Q2: Linear Regression in PyTorch


In [7]:
# Generate synthetic data for linear regression
n_samples = 100
true_weight = 3.5
true_bias = 1.2
X = torch.randn(n_samples, 1)
y = true_weight * X + true_bias + 0.3 * torch.randn(n_samples, 1)

In [8]:
#Visualizing the data
df = pd.DataFrame({
    "x": X.numpy().ravel(),
    "y": y.numpy().ravel(),
})

fig = px.scatter(df, x="x", y="y")
fig.show()

In [9]:
#initialzing linear model
model = LinearRegression(1)

print(f"inital weight {model.linear.weight}, initial bias {model.linear.bias}")

inital weight Parameter containing:
tensor([[0.4801]], requires_grad=True), initial bias Parameter containing:
tensor([0.8415], requires_grad=True)


In [10]:
#unoptimized model's predicitons
pred = model.forward(X)

df["pred"] = pred.detach().numpy().ravel()


fig = px.scatter(df, x="pred", y="y")
fig.show()

In [11]:
#optimizing
criterion = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

# training loop
epoches = 100

#storing loss infor

loss_dict = {"epoch": [x + 1 for x in range(epoches)], 
             "loss" : []}

for i in range(epoches):
    #compute predictions and loss for those predictions
    output = model.forward(X)
    loss = criterion(output, y)

    #compute gradients
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    #store loss for epoch
    loss_dict["loss"].append(float(loss.detach().numpy()))


In [12]:
#optimized model's loss
df2 = pd.DataFrame({
    "epoches" : loss_dict["epoch"],
    "loss": loss_dict["loss"]
})


fig = px.scatter(df2, x="epoches", y = "loss")
fig.show()

In [13]:
#understanding the optimized model

print(f"optimized model's weights {model.linear.weight}")
print(f"optimized model's bias {model.linear.bias}")

optimized model's weights Parameter containing:
tensor([[3.0776]], requires_grad=True)
optimized model's bias Parameter containing:
tensor([1.2121], requires_grad=True)


In [14]:
intercept = torch.tensor(np.ones((100, 1)), dtype=torch.float32)
X_analtical = torch.cat((intercept, X), 1)
X_t_analytical = X_analtical.transpose(0,1)

beta = (torch.linalg.inv(X_t_analytical@X_analtical))@(X_t_analytical@y)
beta

tensor([[1.2107],
        [3.5035]])

In [15]:
# Compare ground truth, analytical OLS, and NN estimates
results = pd.DataFrame(
    [
        {
            "method": "Synthetic truth",
            "weight": float(true_weight),
            "bias": float(true_bias),
        },
        {
            "method": "OLS closed form",
            "weight": float(beta[1].item()),
            "bias": float(beta[0].item()),
        },
        {
            "method": "NN (trained)",
            "weight": float(model.linear.weight.detach().item()),
            "bias": float(model.linear.bias.detach().item()),
        },
    ]
).round(4)

results

,method,weight,bias
0,Synthetic truth,3.5000,1.2000
1,OLS closed form,3.5035,1.2107
2,NN (trained),3.0776,1.2121


In [16]:
#predcited line vs actual data
custom_weight = model.linear.weight.detach().item()
custom_bias = model.linear.bias.detach().item()

df["trend"] = custom_weight * df["x"] + custom_bias

fig = px.scatter(df, x="x", y="y", title="Data with trend line from optimized NN")
fig.add_traces(px.line(df, x="x", y="trend",  color_discrete_sequence=["orange"]).data)
fig.update_layout(showlegend=False)
fig.show()


#### Comaparing SGD with Adam 

In [17]:
adam_model = LinearRegression(1)
#optimizing
criterion_adam = torch.nn.MSELoss()
optimizer_adam = torch.optim.Adam(adam_model.parameters(), lr = 0.01)

# training loop
epoches = 100

#storing loss infor

adam_loss_dict = {"epoch": [x + 1 for x in range(epoches)], 
             "adam_loss" : []}

for i in range(epoches):
    #compute predictions and loss for those predictions
    output = adam_model.forward(X)
    loss_adam = criterion_adam(output, y)

    #compute gradients
    optimizer_adam.zero_grad()
    loss_adam.backward()
    optimizer_adam.step()

    #store loss for epoch
    adam_loss_dict["adam_loss"].append(float(loss_adam.detach().numpy()))

In [18]:
#optimized model's loss
df2["adam"] = adam_loss_dict["adam_loss"]

fig = go.Figure()
fig.add_scatter(
    x=df2["epoches"],
    y=df2["loss"],
    mode="markers",
    marker=dict(color="#1f77b4"),
    name="SGD loss",
)
fig.add_scatter(
    x=df2["epoches"],
    y=df2["adam"],
    mode="markers",
    line=dict(color="orange"),
    name="Adam loss",
)
fig.update_layout(showlegend=True)
fig.show()